# 04 — Fine Wine Heterogeneity Analysis

**Purpose**: Demonstrate that fine wine is highly heterogeneous — individual wines behave very
differently from each other and from market indices during downturns.

**Data source**: `dev_winefi_raf.ml.ml_latent_prices_historic` (latent prices) + `winefi.ml.ml_unified_trades_tbvm` (trade volume).

**Core LWIN7s**: Salon (1807626), DRC Échezeaux (1028658), Lafite (1011872)

**Additional LWIN7s**: Soldera (1226504), Jacques Selosse (1226155), Armand Rousseau Chambertin (1057005), Sassicaia (1102037), Masseto (1160743), Arnoux-Lachaux (1018783), Château Figeac (1009769)

## Sections
1. Environment setup
2. MotherDuck connection & schema discovery
3. Load latent price series per LWIN7
4. Load trade volume per LWIN7
5. Chart 1: Price series (2000 onwards)
6. Chart 2: Trade volume by wine
7. Chart 3: GFC drawdown comparison (core 3 wines vs S&P 500)
8. Chart 4: Extended GFC comparison (2006–2014) showing recovery paths
9. Chart 5: Stress period performance
10. Data quality assertions

## 1. Environment Setup

In [ ]:
import sys
from pathlib import Path

# ---------------------------------------------------------------------------
# Ensure the repo root is in sys.path so project modules can be imported
# regardless of whether this notebook is opened from the repo root or the
# notebook directory itself.
# ---------------------------------------------------------------------------
def _find_repo_root(start: Path) -> Path:
    for parent in [start, *start.parents]:
        if (parent / 'pyproject.toml').exists() or (parent / '.git').exists():
            return parent
    raise RuntimeError(
        'Could not find repo root (no pyproject.toml or .git found). '
        f'Searched from: {start}'
    )

_repo_root = str(_find_repo_root(Path.cwd()))
if _repo_root not in sys.path:
    sys.path.insert(0, _repo_root)

print(f'Repo root: {_repo_root}')

In [ ]:
import os
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
import yfinance as yf
from pathlib import Path

warnings.filterwarnings("ignore")

plt.rcParams.update({'figure.facecolor': 'white', 'axes.facecolor': 'white'})

# ---------------------------------------------------------------------------
# Paths — notebook lives in projects/correlation-diversification/notebooks/
# ---------------------------------------------------------------------------
NOTEBOOK_DIR = Path("__file__").resolve().parent
PROJECT_DIR  = NOTEBOOK_DIR.parent
DATA_DIR     = PROJECT_DIR / "data"
IMAGES_DIR   = PROJECT_DIR / "images" / "heterogeneity"
IMAGES_DIR.mkdir(parents=True, exist_ok=True)

LIVEX_CSV_PATH = DATA_DIR / "liv-ex_index_history.csv"

# ---------------------------------------------------------------------------
# LWIN7 universe
# ---------------------------------------------------------------------------
CORE_LWIN7S = {
    "1807626": "Salon",
    "1028658": "DRC Échezeaux",
    "1011872": "Lafite",
}

ADDITIONAL_LWIN7S = {
    "1226504": "Soldera",
    "1226155": "Jacques Selosse",
    "1057005": "Rousseau Chambertin",
    "1102037": "Sassicaia",
    "1160743": "Masseto",
    "1018783": "Arnoux-Lachaux",
    "1009769": "Château Figeac",
}

ALL_LWIN7S = {**CORE_LWIN7S, **ADDITIONAL_LWIN7S}

# ---------------------------------------------------------------------------
# WineFi brand colours (project palette)
# ---------------------------------------------------------------------------
WINEFI_COLOURS = [
    '#9437ff',  # purple  — primary
    '#83D483',  # mantis
    '#FFD166',  # sunglow
    '#F78C6B',  # coral
    '#4D87D0',  # blue
    '#EF476F',  # red
    '#06D6A0',  # emerald
    '#C23FB7',  # pink/purple
    '#4A4A68',  # slate
]

# Assign colours: core 3 first, then additional
WINE_COLOURS = {
    "Salon":               WINEFI_COLOURS[0],  # purple
    "DRC Échezeaux":       WINEFI_COLOURS[4],  # blue
    "Lafite":              WINEFI_COLOURS[5],  # red
    "Soldera":             WINEFI_COLOURS[1],  # mantis
    "Jacques Selosse":     WINEFI_COLOURS[2],  # sunglow
    "Rousseau Chambertin": WINEFI_COLOURS[3],  # coral
    "Sassicaia":           WINEFI_COLOURS[6],  # emerald
    "Masseto":             WINEFI_COLOURS[7],  # pink/purple
    "Arnoux-Lachaux":      WINEFI_COLOURS[8],  # slate
    "Château Figeac":      WINEFI_COLOURS[0],  # purple (reuse)
}

LIVEX_COLOR  = '#83D483'
SP500_COLOR  = '#FFD166'
STRESS_SHADE = '#4A4A68'

STRESS_PERIODS = [
    ("2008-09-01", "2009-03-31", "GFC"),
    ("2020-02-01", "2020-04-30", "COVID"),
    ("2022-01-01", "2022-12-31", "Rate rises"),
]

GFC_START = pd.Timestamp("2008-09-01")
GFC_END   = pd.Timestamp("2009-03-31")

VINTAGE_CUTOFF = 1980  # filter: only vintages >= 1980

print("Data dir:        ", DATA_DIR)
print("Images dir:      ", IMAGES_DIR)
print("Liv-ex CSV exists:", LIVEX_CSV_PATH.exists())
print(f"Total LWIN7s:     {len(ALL_LWIN7S)} ({len(CORE_LWIN7S)} core + {len(ADDITIONAL_LWIN7S)} additional)")

## 2. MotherDuck Connection & Schema Discovery

Connect to MotherDuck and discover the schema of:
- `dev_winefi_raf.ml.ml_latent_prices_historic` — latent price series (primary data source)
- `winefi.ml.ml_unified_trades_tbvm` — raw trades (for volume charts)

**Schema discovery always runs before any analysis queries.** Never assume column names.

In [ ]:
MD_TOKEN = os.getenv("motherduck_token") or os.getenv("MOTHERDUCK_TOKEN")

if MD_TOKEN:
    import duckdb
    con = duckdb.connect("md:")
    print("Connected to MotherDuck")
    DATA_SOURCE = "MotherDuck (real)"
else:
    con = None
    DATA_SOURCE = "simulated (no MotherDuck token)"
    print("No MotherDuck token — will use simulated data.")

In [ ]:
# Schema discovery: ml_latent_prices_historic
if con is not None:
    latent_schema = con.execute("""
        SELECT
            column_name,
            data_type,
            is_nullable
        FROM information_schema.columns
        WHERE table_catalog = 'dev_winefi_raf'
          AND table_schema  = 'ml'
          AND table_name    = 'ml_latent_prices_historic'
        ORDER BY ordinal_position
    """).df()
    print("=== dev_winefi_raf.ml.ml_latent_prices_historic ===")
    print(f"Columns discovered: {len(latent_schema)}")
    display(latent_schema)
else:
    # Simulate schema
    latent_schema = pd.DataFrame({
        "column_name": ["lwin11", "yyyymm", "price_latent", "region"],
        "data_type":   ["VARCHAR", "INTEGER", "DOUBLE",      "VARCHAR"],
        "is_nullable": ["NO",      "NO",      "YES",         "YES"],
    })
    print("=== dev_winefi_raf.ml.ml_latent_prices_historic (simulated schema) ===")
    display(latent_schema)

In [ ]:
# Schema discovery: ml_unified_trades_tbvm (for trade volume)
if con is not None:
    trades_schema = con.execute("""
        SELECT
            column_name,
            data_type,
            is_nullable
        FROM information_schema.columns
        WHERE table_catalog = 'winefi'
          AND table_schema  = 'ml'
          AND table_name    = 'ml_unified_trades_tbvm'
        ORDER BY ordinal_position
    """).df()
    print("=== winefi.ml.ml_unified_trades_tbvm ===")
    print(f"Columns discovered: {len(trades_schema)}")
    display(trades_schema)
else:
    trades_schema = pd.DataFrame({
        "column_name": ["trade_date", "lwin7", "lwin11", "price_gbp", "quantity", "bottle_size", "vintage"],
        "data_type":   ["DATE",        "VARCHAR", "VARCHAR", "DOUBLE",  "DOUBLE",   "DOUBLE",      "INTEGER"],
        "is_nullable": ["NO",          "YES",     "YES",     "YES",     "YES",      "YES",         "YES"],
    })
    print("=== winefi.ml.ml_unified_trades_tbvm (simulated schema) ===")
    display(trades_schema)

In [ ]:
# ---------------------------------------------------------------------------
# Dynamic column detection helpers
# ---------------------------------------------------------------------------
def first_matching_col(schema_df, *patterns):
    """Return first actual column name whose lowercase form contains any pattern."""
    for _, row in schema_df.iterrows():
        col_lower = row["column_name"].lower()
        for p in patterns:
            if p.lower() in col_lower:
                return row["column_name"]
    return None

# --- Latent prices table ---
LATENT_LWIN11_COL  = first_matching_col(latent_schema, "lwin11", "lwin_11")
LATENT_YYYYMM_COL  = first_matching_col(latent_schema, "yyyymm", "year_month", "yearmonth", "date", "time")
LATENT_PRICE_COL   = first_matching_col(latent_schema, "price_latent", "latent_price", "price")
LATENT_REGION_COL  = first_matching_col(latent_schema, "region")

print("=== Detected columns for ml_latent_prices_historic ===")
print(f"  LWIN11 column: {LATENT_LWIN11_COL}")
print(f"  Date column:   {LATENT_YYYYMM_COL}")
print(f"  Price column:  {LATENT_PRICE_COL}")
print(f"  Region column: {LATENT_REGION_COL}")

if LATENT_LWIN11_COL is None:
    raise ValueError(f"No lwin11 column found. Columns: {latent_schema['column_name'].tolist()}")
if LATENT_YYYYMM_COL is None:
    raise ValueError(f"No date/yyyymm column found. Columns: {latent_schema['column_name'].tolist()}")
if LATENT_PRICE_COL is None:
    raise ValueError(f"No price column found. Columns: {latent_schema['column_name'].tolist()}")

# --- Trades table column detection ---
TRADES_DATE_COL   = first_matching_col(trades_schema, "trade_date", "transaction_date", "date", "time")
TRADES_LWIN7_COL  = first_matching_col(trades_schema, "lwin7", "lwin_7")
TRADES_LWIN11_COL = first_matching_col(trades_schema, "lwin11", "lwin_11")
TRADES_QTY_COL    = first_matching_col(trades_schema, "quantity", "qty", "volume", "bottles", "amount")
TRADES_VINTAGE_COL = first_matching_col(trades_schema, "vintage", "year")
TRADES_BOTTLE_COL = first_matching_col(trades_schema, "bottle_size", "format", "bottle")

print("\n=== Detected columns for ml_unified_trades_tbvm ===")
print(f"  Date column:    {TRADES_DATE_COL}")
print(f"  LWIN7 column:   {TRADES_LWIN7_COL}")
print(f"  LWIN11 column:  {TRADES_LWIN11_COL}")
print(f"  Qty column:     {TRADES_QTY_COL}")
print(f"  Vintage column: {TRADES_VINTAGE_COL}")
print(f"  Bottle column:  {TRADES_BOTTLE_COL}")

# Build LWIN7 SQL expression for trades table
if TRADES_LWIN7_COL:
    TRADES_LWIN7_EXPR = f'CAST("{TRADES_LWIN7_COL}" AS VARCHAR)'
elif TRADES_LWIN11_COL:
    TRADES_LWIN7_EXPR = f'LEFT(CAST("{TRADES_LWIN11_COL}" AS VARCHAR), 7)'
else:
    raise ValueError("No LWIN identifier found in trades table.")

# Vintage filter for trades
if TRADES_VINTAGE_COL:
    TRADES_VINTAGE_FILTER = f'AND CAST("{TRADES_VINTAGE_COL}" AS INTEGER) >= {VINTAGE_CUTOFF}'
elif TRADES_LWIN11_COL:
    # Extract vintage from lwin11 (chars 8-11)
    TRADES_VINTAGE_FILTER = f'AND CAST(SUBSTRING(LPAD(CAST("{TRADES_LWIN11_COL}" AS VARCHAR), 11, \'0\'), 8, 4) AS INTEGER) >= {VINTAGE_CUTOFF}'
else:
    print("WARNING: No vintage column — vintage filter will NOT be applied.")
    TRADES_VINTAGE_FILTER = ""

print(f"  Trades LWIN7 expr:      {TRADES_LWIN7_EXPR}")
print(f"  Trades vintage filter:  {TRADES_VINTAGE_FILTER or '(none)'}")

## 3. Load Latent Price Series per LWIN7

Query `dev_winefi_raf.ml.ml_latent_prices_historic` for all 10 LWIN7s.

The latent prices table stores data at LWIN11 (wine-vintage) level. We:
1. Derive LWIN7 = `LEFT(lwin11, 7)`
2. Derive vintage = `SUBSTRING(lwin11, 8, 4)` and filter to >= 1980
3. Aggregate to a LWIN7-month level median latent price

Falls back to clearly-labelled simulated data when no MotherDuck token is present.

In [ ]:
ALL_IN_LIST = ", ".join(f"'{lw}'" for lw in ALL_LWIN7S.keys())

# Build the SQL — aggregate latent price to LWIN7 + month level
# yyyymm is stored as integer (e.g. 200001 = Jan 2000); cast to DATE
if con is not None:
    latent_sql = f"""
        SELECT
            LEFT(CAST("{LATENT_LWIN11_COL}" AS VARCHAR), 7)                AS lwin7,
            CAST(
                CONCAT(
                    LEFT(CAST("{LATENT_YYYYMM_COL}" AS VARCHAR), 4),
                    '-',
                    RIGHT(CAST("{LATENT_YYYYMM_COL}" AS VARCHAR), 2),
                    '-01'
                ) AS DATE
            )                                                               AS month,
            MEDIAN(CAST("{LATENT_PRICE_COL}" AS DOUBLE))                   AS price_latent,
            COUNT(DISTINCT CAST("{LATENT_LWIN11_COL}" AS VARCHAR))         AS n_vintages
        FROM dev_winefi_raf.ml.ml_latent_prices_historic
        WHERE LEFT(CAST("{LATENT_LWIN11_COL}" AS VARCHAR), 7) IN ({ALL_IN_LIST})
          AND CAST(
                SUBSTRING(LPAD(CAST("{LATENT_LWIN11_COL}" AS VARCHAR), 11, '0'), 8, 4)
              AS INTEGER) >= {VINTAGE_CUTOFF}
          AND "{LATENT_YYYYMM_COL}" >= 200001
        GROUP BY 1, 2
        ORDER BY 1, 2
    """
    print("Latent prices SQL:")
    print(latent_sql)
    try:
        latent_raw = con.execute(latent_sql).df()
        latent_raw["month"] = pd.to_datetime(latent_raw["month"])
        latent_raw["lwin7"] = latent_raw["lwin7"].astype(str).str.strip()
        print(f"\nLoaded {len(latent_raw)} rows from MotherDuck")
        print(f"LWIN7s present: {sorted(latent_raw['lwin7'].unique().tolist())}")
    except Exception as e:
        print(f"MotherDuck query failed: {e} — falling back to simulated data.")
        DATA_SOURCE = "simulated (MotherDuck error)"
        latent_raw = None
else:
    latent_raw = None
    print("No MotherDuck token — will simulate latent prices.")

In [ ]:
# ---------------------------------------------------------------------------
# Simulated fallback — clearly labelled, deterministic (seeded)
# ---------------------------------------------------------------------------
def _simulate_latent_prices(lwin7: str, label: str, seed: int) -> pd.DataFrame:
    """Generate plausible deterministic simulated monthly latent price series."""
    rng   = np.random.default_rng(seed)
    dates = pd.date_range("2000-01-01", "2025-12-01", freq="MS")
    n     = len(dates)

    # Approximate price levels and GFC drawdowns per wine
    _base_prices = {
        "1807626": 1600, "1028658": 1800, "1011872": 300,
        "1226504": 600,  "1226155": 150,  "1057005": 900,
        "1102037": 200,  "1160743": 450,  "1018783": 350, "1009769": 120,
    }
    _gfc_drawdowns = {
        "1807626": -0.04, "1028658": -0.05, "1011872": -0.22,
        "1226504": -0.03, "1226155": -0.12, "1057005": -0.08,
        "1102037": -0.18, "1160743": -0.08, "1018783": -0.06, "1009769": -0.20,
    }

    base  = _base_prices.get(lwin7, 250)
    gfc_d = _gfc_drawdowns.get(lwin7, -0.15)

    gfc_start_idx = int(np.searchsorted(dates.to_numpy(), np.datetime64("2008-09-01")))
    gfc_end_idx   = int(np.searchsorted(dates.to_numpy(), np.datetime64("2009-03-01")))
    covid_s_idx   = int(np.searchsorted(dates.to_numpy(), np.datetime64("2020-02-01")))
    covid_e_idx   = int(np.searchsorted(dates.to_numpy(), np.datetime64("2020-04-01")))
    rate_s_idx    = int(np.searchsorted(dates.to_numpy(), np.datetime64("2022-01-01")))
    rate_e_idx    = int(np.searchsorted(dates.to_numpy(), np.datetime64("2022-12-01")))

    price = np.zeros(n)
    price[0] = base * 0.4
    for i in range(1, n):
        if i < gfc_start_idx:
            drift, noise = 0.004, 0.022
        elif i <= gfc_end_idx:
            drift = gfc_d / max(gfc_end_idx - gfc_start_idx, 1)
            noise = 0.012
        elif i <= gfc_end_idx + 24:
            drift, noise = 0.005, 0.018
        elif covid_s_idx <= i <= covid_e_idx:
            drift, noise = -0.04, 0.015
        elif rate_s_idx <= i <= rate_e_idx:
            drift, noise = -0.01, 0.020
        else:
            drift, noise = 0.002, 0.018
        price[i] = max(price[i - 1] * (1 + drift + rng.normal(0, noise)), base * 0.10)

    return pd.DataFrame({
        "lwin7":         lwin7,
        "month":         dates,
        "price_latent":  price,
        "n_vintages":    rng.integers(1, 6, size=n),
    })


if latent_raw is None or len(latent_raw) == 0:
    print(f"Using simulated latent prices (data source: {DATA_SOURCE})")
    latent_raw = pd.concat(
        [
            _simulate_latent_prices(lwin7, label, seed=i * 17 + 3)
            for i, (lwin7, label) in enumerate(ALL_LWIN7S.items())
        ],
        ignore_index=True,
    )
    latent_raw["month"] = pd.to_datetime(latent_raw["month"])

latent_raw["wine_name"] = latent_raw["lwin7"].map(ALL_LWIN7S)

print(f"Latent price data ({DATA_SOURCE}): {latent_raw.shape}")
print(f"Date range: {latent_raw['month'].min().date()} → {latent_raw['month'].max().date()}")
print(f"\nRows per wine:")
print(latent_raw.groupby(["lwin7", "wine_name"])["price_latent"].count().rename("n_months").to_string())

In [ ]:
# ---------------------------------------------------------------------------
# Pivot to wide format: rows = month, cols = wine_name
# ---------------------------------------------------------------------------
price_wide = latent_raw.pivot_table(
    index="month", columns="wine_name", values="price_latent", aggfunc="median"
)
price_wide.columns.name = None
price_wide.index.name   = "date"

# Reindex to complete monthly calendar (first of month)
full_idx = pd.date_range(
    start=price_wide.index.min(),
    end=price_wide.index.max(),
    freq="MS",
)
price_wide = price_wide.reindex(full_idx)

# Forward-fill for continuity in charts; keep raw for volume
price_ffill = price_wide.ffill()

# Filter to 2000+
CHART_START = pd.Timestamp("2000-01-01")
price_wide  = price_wide[price_wide.index >= CHART_START]
price_ffill = price_ffill[price_ffill.index >= CHART_START]

wines_present = [w for w in ALL_LWIN7S.values() if w in price_ffill.columns]
core_wines    = [w for w in CORE_LWIN7S.values() if w in price_ffill.columns]

print(f"Price series shape:  {price_ffill.shape}")
print(f"Wines present:       {wines_present}")
print(f"Core wines:          {core_wines}")
price_ffill.tail(3)

## 4. Load Trade Volume per LWIN7

Monthly trade count for all wines from `winefi.ml.ml_unified_trades_tbvm`.
Vintage >= 1980 filter applied. Only 750ml bottles where bottle size available.

In [ ]:
# Build bottle filter
if TRADES_BOTTLE_COL:
    BOTTLE_FILTER = f'AND CAST("{TRADES_BOTTLE_COL}" AS DOUBLE) = 750'
else:
    BOTTLE_FILTER = ""

if con is not None and TRADES_DATE_COL is not None:
    vol_sql = f"""
        SELECT
            DATE_TRUNC('month', CAST("{TRADES_DATE_COL}" AS DATE))  AS month,
            {TRADES_LWIN7_EXPR}                                      AS lwin7,
            COUNT(*)                                                  AS trade_count,
            {f'SUM(CAST("{TRADES_QTY_COL}" AS DOUBLE))' if TRADES_QTY_COL else 'NULL::DOUBLE'}
                                                                     AS total_qty
        FROM winefi.ml.ml_unified_trades_tbvm
        WHERE {TRADES_LWIN7_EXPR} IN ({ALL_IN_LIST})
          AND CAST("{TRADES_DATE_COL}" AS DATE) >= '2000-01-01'
          {BOTTLE_FILTER}
          {TRADES_VINTAGE_FILTER}
        GROUP BY 1, 2
        ORDER BY 1, 2
    """
    print("Trade volume SQL:")
    print(vol_sql)
    try:
        volume_raw = con.execute(vol_sql).df()
        volume_raw["month"] = pd.to_datetime(volume_raw["month"])
        volume_raw["lwin7"] = volume_raw["lwin7"].astype(str).str.strip()
        print(f"\nLoaded {len(volume_raw)} rows of trade volume from MotherDuck")
    except Exception as e:
        print(f"Volume query failed: {e} — simulating.")
        volume_raw = None
else:
    volume_raw = None
    print("No MotherDuck connection — will simulate trade volume.")

In [ ]:
# Simulated fallback for trade volume
def _simulate_volume(lwin7: str, label: str, seed: int) -> pd.DataFrame:
    rng   = np.random.default_rng(seed + 100)
    dates = pd.date_range("2000-01-01", "2025-12-01", freq="MS")
    n     = len(dates)
    # Sparse wines get lower base counts
    base  = {"1807626": 8, "1028658": 3, "1011872": 40,
             "1226504": 2, "1226155": 2, "1057005": 5,
             "1102037": 15, "1160743": 10, "1018783": 6, "1009769": 20}.get(lwin7, 10)
    counts = rng.integers(0, base * 3, size=n)
    df = pd.DataFrame({"month": dates, "lwin7": lwin7, "trade_count": counts, "total_qty": counts * 6.0})
    return df[df["trade_count"] > 0]


if volume_raw is None or len(volume_raw) == 0:
    volume_raw = pd.concat(
        [
            _simulate_volume(lwin7, label, seed=i * 7 + 11)
            for i, (lwin7, label) in enumerate(ALL_LWIN7S.items())
        ],
        ignore_index=True,
    )

volume_raw["wine_name"] = volume_raw["lwin7"].map(ALL_LWIN7S)

# Pivot to wide format
volume_wide = volume_raw.pivot_table(
    index="month", columns="wine_name", values="trade_count", aggfunc="sum"
)
volume_wide.columns.name = None
volume_wide.index.name   = "date"
volume_wide = volume_wide.reindex(full_idx).fillna(0)
volume_wide = volume_wide[volume_wide.index >= CHART_START]

print(f"Volume data ({DATA_SOURCE}): {volume_wide.shape}")
volume_wide.tail(3)

## 4.5 Comparison Assets

Load S&P 500 Total Return (USD, GBP-adjusted) and Liv-ex Fine Wine 100 from CSV.

In [ ]:
# Liv-ex Fine Wine 100 from CSV
livex_csv_raw = pd.read_csv(LIVEX_CSV_PATH)
livex_csv_raw["date"] = pd.to_datetime(livex_csv_raw["date"])
livex_csv_raw = (
    livex_csv_raw
    .dropna(subset=["date"])
    .sort_values("date")
    .set_index("date")
)

LIVEX_100_COL = "Liv-ex Fine Wine 100"
livex_100 = livex_csv_raw[LIVEX_100_COL].dropna().resample("MS").last().dropna()

print(f"Liv-ex Fine Wine 100: {len(livex_100)} months, "
      f"{livex_100.index.min().date()} → {livex_100.index.max().date()}")

In [ ]:
# S&P 500 Total Return (USD) + GBP adjustment
sp500_usd = yf.download("^SP500TR", start="2000-01-01", progress=False, auto_adjust=False)["Close"]
if isinstance(sp500_usd, pd.DataFrame):
    sp500_usd = sp500_usd.squeeze()
sp500_usd = sp500_usd.resample("MS").last()

usdgbp = yf.download("USDGBP=X", start="2000-01-01", progress=False, auto_adjust=False)["Close"]
if isinstance(usdgbp, pd.DataFrame):
    usdgbp = usdgbp.squeeze()
usdgbp = usdgbp.resample("MS").last()

sp500_gbp = (sp500_usd * usdgbp).dropna()

print(f"S&P 500 (GBP-adjusted): {len(sp500_gbp)} months, "
      f"{sp500_gbp.index.min().date()} → {sp500_gbp.index.max().date()}")

## 5. Chart 1: Price Series (2000 Onwards)

One subplot per wine. Latent prices forward-filled. Stress periods shaded.

In [ ]:
n_wines = len(wines_present)
fig, axes = plt.subplots(
    n_wines, 1,
    figsize=(14, 4 * n_wines),
    sharex=True,
)
if n_wines == 1:
    axes = [axes]

fig.suptitle(
    "Fine Wine Latent Price Series (GBP) — 2000 Onwards",
    fontsize=13, fontweight="bold", y=1.005,
)

for ax, wine_name in zip(axes, wines_present):
    color     = WINE_COLOURS.get(wine_name, WINEFI_COLOURS[0])
    price_s   = price_ffill[wine_name].dropna()
    price_raw_s = price_wide[wine_name]

    # Shade stress periods
    for p_start, p_end, p_label in STRESS_PERIODS:
        ax.axvspan(pd.Timestamp(p_start), pd.Timestamp(p_end),
                   alpha=0.09, color=STRESS_SHADE, zorder=0)

    # Main price line (solid where data, dashed where ffill only)
    has_data = price_raw_s.notna()
    solid_s  = price_ffill[wine_name].where(has_data)
    gap_s    = price_ffill[wine_name].where(~has_data)

    ax.plot(solid_s.index, solid_s.values, color=color, linewidth=2.0, label=wine_name)
    ax.plot(gap_s.index, gap_s.values, color=color, linewidth=1.0,
            linestyle="--", alpha=0.4, label="_nolegend_")

    ax.set_ylabel("Latent price (GBP)", fontsize=9)
    ax.set_title(wine_name, fontsize=11, loc="left", color=color, fontweight="bold")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"£{x:,.0f}"))
    ax.legend(fontsize=8, loc="upper left")
    ax.grid(axis="y", alpha=0.3)

    # Stress period labels at bottom of each subplot
    ylim    = ax.get_ylim()
    y_label = ylim[0] + (ylim[1] - ylim[0]) * 0.04
    for p_start, p_end, p_label in STRESS_PERIODS:
        ts, te = pd.Timestamp(p_start), pd.Timestamp(p_end)
        ax.text(ts + (te - ts) / 2, y_label, p_label,
                ha="center", va="bottom", fontsize=7, color=STRESS_SHADE, alpha=0.8)

axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
fig.text(
    0.5, -0.01,
    f"Source: WineFi latent price model (dev_winefi_raf.ml.ml_latent_prices_historic). "
    f"Vintage >= {VINTAGE_CUTOFF} only. Dashed = forward-filled (no data month). "
    f"Data: {DATA_SOURCE}.",
    ha="center", fontsize=7, color="gray",
)
plt.tight_layout()

out_path = IMAGES_DIR / "wine_price_series.png"
fig.savefig(out_path, dpi=150, bbox_inches="tight")
print(f"Saved → {out_path}")
plt.show()

## 6. Chart 2: Trade Volume by Wine

Monthly trade count per wine. Stress periods shaded.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

for wine_name in wines_present:
    if wine_name not in volume_wide.columns:
        continue
    color   = WINE_COLOURS.get(wine_name, WINEFI_COLOURS[0])
    vol_s   = volume_wide[wine_name].replace(0, np.nan).dropna()
    lw      = 2.0 if wine_name in core_wines else 1.2
    alpha   = 0.95 if wine_name in core_wines else 0.65
    ax.plot(vol_s.index, vol_s.values, color=color, linewidth=lw, alpha=alpha, label=wine_name)

for p_start, p_end, p_label in STRESS_PERIODS:
    ts, te = pd.Timestamp(p_start), pd.Timestamp(p_end)
    ax.axvspan(ts, te, alpha=0.09, color=STRESS_SHADE, zorder=0)
    ylim    = ax.get_ylim()
    y_label = ylim[0] + (ylim[1] - ylim[0]) * 0.92
    ax.text(ts + (te - ts) / 2, y_label, p_label,
            ha="center", va="top", fontsize=7, color=STRESS_SHADE, alpha=0.8)

ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.set_ylabel("Monthly trade count (# transactions)", fontsize=10)
ax.set_title(
    "Monthly Trade Volume by Wine — Core Wines (bold) vs Additional",
    fontsize=11, fontweight="bold",
)
ax.legend(fontsize=8, ncol=2)
ax.grid(axis="y", alpha=0.3)
fig.text(
    0.5, -0.02,
    f"Source: WineFi transaction data (winefi.ml.ml_unified_trades_tbvm). "
    f"Vintage >= {VINTAGE_CUTOFF}. 750 ml only where available. Data: {DATA_SOURCE}.",
    ha="center", fontsize=7, color="gray",
)
plt.tight_layout()

out_path = IMAGES_DIR / "wine_trade_volume.png"
fig.savefig(out_path, dpi=150, bbox_inches="tight")
print(f"Saved → {out_path}")
plt.show()

## 7. Chart 3: GFC Drawdown Comparison — Core 3 Wines vs S&P 500

All series indexed to 100 at January 2007. Shows divergence of individual wines
vs equity market during the 2008 GFC.

In [ ]:
INDEX_BASE = pd.Timestamp("2007-01-01")
GFC_PLOT_START = pd.Timestamp("2006-01-01")
GFC_PLOT_END   = pd.Timestamp("2012-12-01")


def index_to_100(series: pd.Series, base_date: pd.Timestamp) -> pd.Series | None:
    """Rebase series to 100 at base_date (nearest available)."""
    candidates = series[
        (series.index >= base_date) & (series.index <= base_date + pd.DateOffset(months=2))
    ].dropna()
    if candidates.empty:
        return None
    base_val = candidates.iloc[0]
    if base_val == 0 or np.isnan(base_val):
        return None
    return series / base_val * 100


gfc_series = {}
for wine_name in core_wines:
    gfc_series[wine_name] = price_ffill[wine_name].dropna()
gfc_series["Liv-ex 100"]   = livex_100
gfc_series["S&P 500 (GBP)"] = sp500_gbp

fig, ax = plt.subplots(figsize=(14, 6))

for name, series in gfc_series.items():
    s = series[(series.index >= GFC_PLOT_START) & (series.index <= GFC_PLOT_END)].dropna()
    if len(s) < 3:
        print(f"  Skipping '{name}': only {len(s)} points")
        continue
    s_idx = index_to_100(s, INDEX_BASE)
    if s_idx is None:
        print(f"  Skipping '{name}': cannot rebase at {INDEX_BASE.date()}")
        continue

    if name in core_wines:
        color, lw, ls, alpha_val = WINE_COLOURS[name], 2.2, "-", 1.0
    elif "Liv-ex" in name:
        color, lw, ls, alpha_val = LIVEX_COLOR, 1.5, "--", 0.3
    else:
        color, lw, ls, alpha_val = SP500_COLOR, 2.0, "--", 0.85

    ax.plot(s_idx.index, s_idx.values, color=color,
            linewidth=lw, linestyle=ls, alpha=alpha_val, label=name)

ax.axvspan(GFC_START, GFC_END, alpha=0.12, color="#EF476F", zorder=0, label="GFC (Sep 08–Mar 09)")
ax.axhline(100, color=STRESS_SHADE, linewidth=0.7, linestyle=":")

ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.set_ylabel("Indexed return (Jan 2007 = 100)", fontsize=10)
ax.set_title(
    "GFC Drawdown: Core Wines vs S&P 500 (indexed to 100 at Jan 2007)",
    fontsize=11, fontweight="bold",
)
ax.legend(fontsize=9, loc="lower left")
ax.grid(axis="y", alpha=0.3)
fig.text(
    0.5, -0.02,
    f"Source: WineFi latent prices (GBP); S&P 500 Total Return USD×USDGBP=X; "
    f"Liv-ex 100 (faint dashed). Data: {DATA_SOURCE}.",
    ha="center", fontsize=7, color="gray",
)
plt.tight_layout()

out_path = IMAGES_DIR / "gfc_drawdown_comparison.png"
fig.savefig(out_path, dpi=150, bbox_inches="tight")
print(f"Saved → {out_path}")
plt.show()

## 8. Chart 4: Extended GFC Comparison (2006–2014) — Recovery Paths

All 10 wines indexed to 100 at January 2007. Extended window shows diverging recovery
trajectories after the GFC trough. Faint Liv-ex 100 and S&P 500 benchmark overlays.

In [ ]:
GFC_EXT_START = pd.Timestamp("2006-01-01")
GFC_EXT_END   = pd.Timestamp("2014-12-01")

fig, ax = plt.subplots(figsize=(14, 7))

for wine_name in wines_present:
    color = WINE_COLOURS.get(wine_name, WINEFI_COLOURS[0])
    s = price_ffill[wine_name][
        (price_ffill[wine_name].index >= GFC_EXT_START) &
        (price_ffill[wine_name].index <= GFC_EXT_END)
    ].dropna()
    if len(s) < 3:
        continue
    s_idx = index_to_100(s, INDEX_BASE)
    if s_idx is None:
        continue

    is_core = wine_name in core_wines
    lw      = 2.2 if is_core else 1.4
    ls      = "-"  if is_core else "--"
    alpha   = 1.0  if is_core else 0.75

    # Annotate GFC trough return
    gfc_window = s_idx[
        (s_idx.index >= GFC_START) & (s_idx.index <= GFC_END)
    ].dropna()
    trough_val = gfc_window.min() if len(gfc_window) > 0 else np.nan
    trough_str = f" ({trough_val - 100:+.0f}%)" if not np.isnan(trough_val) else ""

    ax.plot(s_idx.index, s_idx.values, color=color,
            linewidth=lw, linestyle=ls, alpha=alpha,
            label=f"{wine_name}{trough_str}")

# Benchmark overlays (faint)
for bm_name, bm_series, bm_color in [
    ("Liv-ex 100",   livex_100, "dimgray"),
    ("S&P 500 (GBP)", sp500_gbp, SP500_COLOR),
]:
    s = bm_series[(bm_series.index >= GFC_EXT_START) & (bm_series.index <= GFC_EXT_END)].dropna()
    if len(s) >= 3:
        s_idx = index_to_100(s, INDEX_BASE)
        if s_idx is not None:
            ax.plot(s_idx.index, s_idx.values, color=bm_color,
                    linewidth=1.5, linestyle="-", alpha=0.25, label=f"{bm_name} (benchmark)")

ax.axvspan(GFC_START, GFC_END, alpha=0.12, color="#EF476F", zorder=0, label="GFC (Sep 08–Mar 09)")
ax.axhline(100, color=STRESS_SHADE, linewidth=0.7, linestyle=":")

ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.set_ylabel("Indexed return (Jan 2007 = 100)", fontsize=10)
ax.set_title(
    "GFC & Recovery (2006–2014): All Wines Indexed to 100 at Jan 2007\n"
    "(Core wines solid; additional dashed; legend shows GFC trough drawdown)",
    fontsize=11, fontweight="bold",
)
ax.legend(fontsize=8, loc="lower left", ncol=2)
ax.grid(axis="y", alpha=0.3)
fig.text(
    0.5, -0.02,
    f"Source: WineFi latent prices; S&P 500 Total Return (GBP-adjusted); Liv-ex 100 CSV. "
    f"Vintage >= {VINTAGE_CUTOFF}. Data: {DATA_SOURCE}.",
    ha="center", fontsize=7, color="gray",
)
plt.tight_layout()

out_path = IMAGES_DIR / "gfc_extended_comparison.png"
fig.savefig(out_path, dpi=150, bbox_inches="tight")
print(f"Saved → {out_path}")
plt.show()

## 9. Chart 5: Stress Period Performance

Total return (%) per wine and benchmark across three crisis events:
GFC (Sep 2008–Mar 2009), COVID (Feb–Apr 2020), Rate rises (Jan–Dec 2022).

In [ ]:
def period_return(series: pd.Series, start: str, end: str) -> float:
    """Total % return of series between start and end (inclusive)."""
    window = series[
        (series.index >= pd.Timestamp(start)) &
        (series.index <= pd.Timestamp(end))
    ].dropna()
    if len(window) < 2:
        return np.nan
    return float((window.iloc[-1] - window.iloc[0]) / window.iloc[0] * 100)


# Build asset dict: all wines + benchmarks
stress_assets: dict[str, pd.Series] = {}
for wine_name in wines_present:
    stress_assets[wine_name] = price_ffill[wine_name].dropna()
stress_assets["Liv-ex 100"]    = livex_100
stress_assets["S&P 500 (GBP)"] = sp500_gbp

stress_rows = []
for p_start, p_end, p_label in STRESS_PERIODS:
    row = {"Period": p_label}
    for name, series in stress_assets.items():
        row[name] = period_return(series, p_start, p_end)
    stress_rows.append(row)

stress_df = pd.DataFrame(stress_rows).set_index("Period")

print("Return (%) during each stress period:")
display(stress_df.round(1))

In [ ]:
all_cols   = list(stress_df.columns)
n_assets   = len(all_cols)
n_periods  = len(stress_df)
x          = np.arange(n_periods)
bar_width  = 0.7 / n_assets

color_list = []
for col in all_cols:
    if col in WINE_COLOURS:
        color_list.append(WINE_COLOURS[col])
    elif "Liv-ex" in col:
        color_list.append(LIVEX_COLOR)
    else:
        color_list.append(SP500_COLOR)

fig, ax = plt.subplots(figsize=(16, 6))

for i, (col, col_color) in enumerate(zip(all_cols, color_list)):
    vals   = stress_df[col].values.astype(float)
    offset = (i - n_assets / 2 + 0.5) * bar_width
    is_bm  = "Liv-ex" in col
    alpha  = 0.35 if is_bm else 0.85
    bars   = ax.bar(
        x + offset, vals, width=bar_width,
        color=col_color, label=col,
        alpha=alpha, edgecolor="white", linewidth=0.4,
    )
    for bar, val in zip(bars, vals):
        if not np.isnan(val):
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                val + (0.5 if val >= 0 else -1.5),
                f"{val:+.0f}%",
                ha="center",
                va="bottom" if val >= 0 else "top",
                fontsize=5, color=STRESS_SHADE,
            )

ax.axhline(0, color=STRESS_SHADE, linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels(stress_df.index, fontsize=11)
ax.set_ylabel("Total return during period (%)", fontsize=10)
ax.set_title(
    "Fine Wine vs Benchmarks: Total Return During Market Stress Periods",
    fontsize=12, fontweight="bold",
)
ax.legend(fontsize=7, loc="lower right", ncol=3)
ax.grid(axis="y", alpha=0.3)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:+.0f}%"))
fig.text(
    0.5, -0.02,
    "GFC: Sep 2008–Mar 2009 | COVID: Feb–Apr 2020 | Rate rises: Jan–Dec 2022. "
    f"S&P 500 GBP-adjusted (×USDGBP=X). Wine: latent prices, ffill. Data: {DATA_SOURCE}.",
    ha="center", fontsize=7, color="gray",
)
plt.tight_layout()

out_path = IMAGES_DIR / "stress_period_performance.png"
fig.savefig(out_path, dpi=150, bbox_inches="tight")
print(f"Saved → {out_path}")
plt.show()

## 10. Data Quality Assertions

All assertions must pass before the notebook is considered complete.

In [ ]:
errors = []

# 1. Required PNG files
required_images = [
    "wine_price_series.png",
    "wine_trade_volume.png",
    "gfc_drawdown_comparison.png",
    "gfc_extended_comparison.png",
    "stress_period_performance.png",
]
for img in required_images:
    path = IMAGES_DIR / img
    if not path.exists():
        errors.append(f"Missing image: {img}")
    elif path.stat().st_size < 5_000:
        errors.append(f"Image appears empty (<5 KB): {img}")

# 2. Core wines present
for wine in ["Salon", "DRC Échezeaux", "Lafite"]:
    if wine not in wines_present:
        errors.append(f"Core wine '{wine}' missing from price series")

# 3. Price series date range
if not price_ffill.empty:
    if price_ffill.index.min().year > 2001:
        errors.append(f"Price series starts {price_ffill.index.min().year}, expected <= 2001")
    if price_ffill.index.max().year < 2020:
        errors.append(f"Price series ends {price_ffill.index.max().year}, expected >= 2020")
else:
    errors.append("price_ffill is empty")

# 4. S&P 500 GBP series
if len(sp500_gbp) < 100:
    errors.append(f"sp500_gbp has only {len(sp500_gbp)} rows (expected >= 100)")

# 5. Livex 100 series
if len(livex_100) < 100:
    errors.append(f"livex_100 has only {len(livex_100)} rows (expected >= 100)")

# 6. Stress period returns for at least 2 assets per period
for period in stress_df.index:
    valid = stress_df.loc[period].notna().sum()
    if valid < 2:
        errors.append(f"Stress period '{period}': only {valid} assets with data (expected >= 2)")

# 7. All 10 LWIN7s present in latent data
found_lwin7s = set(latent_raw["lwin7"].unique())
missing_lwin7s = set(ALL_LWIN7S.keys()) - found_lwin7s
if missing_lwin7s:
    errors.append(f"Missing LWIN7s in latent data: {missing_lwin7s}")

# 8. Vintage filter applied (no latent row should have vintage < 1980 in LWIN11)
if con is not None and LATENT_LWIN11_COL is not None:
    try:
        _v_check = con.execute(f"""
            SELECT COUNT(*) AS cnt
            FROM dev_winefi_raf.ml.ml_latent_prices_historic
            WHERE LEFT(CAST("{LATENT_LWIN11_COL}" AS VARCHAR), 7) IN ({ALL_IN_LIST})
              AND CAST(SUBSTRING(LPAD(CAST("{LATENT_LWIN11_COL}" AS VARCHAR), 11, '0'), 8, 4) AS INTEGER) < {VINTAGE_CUTOFF}
              AND "{LATENT_YYYYMM_COL}" >= 200001
            LIMIT 1
        """).df()["cnt"].iloc[0]
        if _v_check > 0:
            errors.append(f"Vintage filter breach: {_v_check} rows with vintage < {VINTAGE_CUTOFF} still present")
    except Exception:
        pass  # non-critical check

# Report
if errors:
    for err in errors:
        print(f"FAIL: {err}")
    raise AssertionError("Data quality assertions failed — see output above")
else:
    print("All data quality assertions PASSED.")
    print(f"  Images saved ({len(required_images)}): {required_images}")
    print(f"  Wines present ({len(wines_present)}): {wines_present}")
    print(f"  Core wines: {core_wines}")
    print(f"  Price series: {price_ffill.index.min().date()} → {price_ffill.index.max().date()}")
    print(f"  S&P 500 GBP: {len(sp500_gbp)} months")
    print(f"  Liv-ex 100: {len(livex_100)} months")
    print(f"  Data source: {DATA_SOURCE}")

## Summary

| Output | Description | Location |
|--------|-------------|----------|
| `wine_price_series.png` | Latent price series per wine (2000+), stress periods shaded | `images/heterogeneity/` |
| `wine_trade_volume.png` | Monthly trade count by wine, stress periods shaded | `images/heterogeneity/` |
| `gfc_drawdown_comparison.png` | Core 3 wines + S&P 500, indexed to 100 at Jan 2007 | `images/heterogeneity/` |
| `gfc_extended_comparison.png` | All 10 wines: GFC & recovery (2006–2014), indexed to 100 at Jan 2007 | `images/heterogeneity/` |
| `stress_period_performance.png` | Grouped bar: total return % per asset across GFC / COVID / Rate rises | `images/heterogeneity/` |

**Core LWIN7s**: Salon (1807626), DRC Échezeaux (1028658), Lafite (1011872)

**Additional LWIN7s**: Soldera (1226504), Jacques Selosse (1226155), Rousseau Chambertin (1057005), Sassicaia (1102037), Masseto (1160743), Arnoux-Lachaux (1018783), Château Figeac (1009769)

**Data source**: `dev_winefi_raf.ml.ml_latent_prices_historic` (latent prices) — LWIN11 level, aggregated to LWIN7 + month. Vintage >= 1980 filter applied. Falls back to deterministic simulated data without MotherDuck token.

**All prices in GBP.** S&P 500 GBP-adjusted via `USDGBP=X`.